# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print dataset title and description
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.
Each `@id` uniquely identifies an entity in the dataset. We'll list all record sets and their schema briefly.

In [ ]:
# List all available record set `@id`s, field `@id`s, and column `@id`s
print('Available record sets and their fields:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"\nRecord set @id: {rs.id}\n  Name: {getattr(rs, 'name', '(no name)')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {getattr(field, 'name', '(no name)')})")
    if hasattr(rs, 'columns'):
        print("  Columns:")
        for column in getattr(rs, 'columns', []):
            print(f"    - {column.id} (name: {getattr(column, 'name', '(no name)')})")

## Display a record example from a record set
Iterate through a single record to inspect the structure and typical entries (referenced by @id).

In [ ]:
# Example: Display the first record of the first record set
if record_sets:
    record_set_id = record_sets[0].id
    print(f"\nExample record from record set {record_set_id}")
    record_iter = dataset.records(record_set=record_set_id)
    try:
        record = next(record_iter)
        print(record)
    except StopIteration:
        print(f"No records available in {record_set_id}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.
We reference all entities by their `@id` fields, as per the croissant specification.

In [ ]:
# Extract data from each record set
# Use @id of record sets for consistent referencing
# Data will be loaded into dataframes indexed by record set @id
dataframes = {}
for rs in record_sets:
    recs = list(dataset.records(record_set=rs.id))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rs.id] = df
        print(f"Loaded {len(df)} records for record set @id={rs.id}")
        print(f"Columns for {rs.id}: {df.columns.tolist()}")
    else:
        print(f"No records in record set {rs.id}")


### Preview a DataFrame
Let's select a main record set (the one with the most records or with protein information) for further analysis.

In [ ]:
# Choose the main record set (with highest number of rows)
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"Main record set selected for EDA: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes to analyze.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps: filter records by a numeric field, normalize, and group by another field. All fields and columns referenced by their `@id`.

In [ ]:
if dataframes:
    df = dataframes[main_record_set_id]
    # Try to find a numeric field (e.g., molecular weight, "mw" is a common protein column)
    numeric_field_candidates = [
        c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int] or df[c].apply(lambda x: str(x).replace('.', '', 1).isdigit()).sum() > 0
    ]
    # Fallback: guess a column that looks like MW, abundance, etc.
    if not numeric_field_candidates:
        for col in df.columns:
            if any(k in col.lower() for k in ['mw', 'mass', 'weight', 'abundance', 'count']):
                numeric_field_candidates.append(col)
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Numeric field chosen (by @id): {numeric_field}")
        # Coerce to numeric if necessary
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.9) # Use 90th percentile as an example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with '{numeric_field}' > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize field
        mu = filtered_df[numeric_field].mean()
        sigma = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mu) / sigma
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to choose a group field (such as accession, description, or any string/categorical)
        group_field_candidates = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"Grouping field chosen (by @id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            grouped_df = grouped_df.sort_values(by=numeric_field, ascending=False)
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
    else:
        print("No suitable numeric field for EDA.")
else:
    print("No dataframes for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals() and not filtered_df.empty:
    # Histogram of the numeric field after filtering
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of '{numeric_field}' after filtering")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # Scatter plot of normalized field vs. group (if available)
    if 'group_field' in locals() and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        sns.stripplot(x=group_field, y=f"{numeric_field}_normalized", data=filtered_df, jitter=True)
        plt.title(f"Normalized '{numeric_field}' by '{group_field}'")
        plt.ylabel(f"{numeric_field}_normalized")
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the FAIR² mass spectrometry dataset with all entities referenced by `@id`. We:
- Inspected all record sets and fields via their Croissant schema.
- Loaded the main record set into a DataFrame, filtered and normalized numeric measurements.
- Grouped and visualized the tabular protein data.

The dataset enables downstream tasks, such as protein biomarker discovery, by facilitating reproducible exploratory and statistical analyses from standardized FAIR data representations.